<a href="https://colab.research.google.com/github/h-yuri/h-yuri/blob/master/ntitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 【統合版】差分取得パイプライン（Blayn + ticket.shiagel.jp）
# - GitHub Actions 対応（Colab依存排除 / Secrets対応）
# - 取得管理（店舗別 最終取得日）+1日 ～ 昨日まで
# - Fact_時間別：追記（重複は 店舗×日付×時間 で除外）
# - jpholiday：祝日対応（曜日ラベル）
# - Blayn：requests + bs4(lxml)
# - ticket：Playwright（headless）
# ============================================================

import os, re, asyncio, json, time
from datetime import datetime, timedelta

import requests
from bs4 import BeautifulSoup
import jpholiday

from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError


# ============================================================
# 0) 環境変数（GitHub Actions から渡す）
#   - どっちでもOK:
#     (A) workflowで service_account.json を作って SERVICE_ACCOUNT_FILE を渡す
#     (B) Secret にJSON文字列を入れて SERVICE_ACCOUNT_JSON で渡す（このコードがファイル化）
# ============================================================

def env_required(key: str) -> str:
    v = os.environ.get(key)
    if not v:
        raise ValueError(f"環境変数 {key} が未設定です（GitHub Secrets から渡してください）")
    return v

SPREADSHEET_ID = env_required("SPREADSHEET_ID")

SERVICE_ACCOUNT_FILE = os.environ.get("SERVICE_ACCOUNT_FILE", "service_account.json")

# もしファイルが無いなら、JSON文字列のSecretから作る（任意）
SERVICE_ACCOUNT_JSON = os.environ.get("SERVICE_ACCOUNT_JSON", "") or os.environ.get("GCP_SA_JSON", "")
if (not os.path.exists(SERVICE_ACCOUNT_FILE)) and SERVICE_ACCOUNT_JSON:
    with open(SERVICE_ACCOUNT_FILE, "w", encoding="utf-8") as f:
        f.write(SERVICE_ACCOUNT_JSON)
    print("✅ service_account.json を Secret(JSON文字列) から生成しました:", SERVICE_ACCOUNT_FILE)

if not os.path.exists(SERVICE_ACCOUNT_FILE):
    raise FileNotFoundError(f"service_account.json が見つかりません: {SERVICE_ACCOUNT_FILE}")

print("✅ Using SERVICE_ACCOUNT_FILE:", SERVICE_ACCOUNT_FILE)
print("✅ Using SPREADSHEET_ID:", SPREADSHEET_ID)


# ============================================================
# 1) Google Sheets 共通（Fact/取得管理）
# ============================================================

SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

SHEET_FACT  = "Fact_時間別"
SHEET_STATE = "取得管理"

FACT_HEADER = [
    "日付","月","曜日","時間","時間帯区分",
    "売上（税込）","売上（税抜）",
    "ラーメン数（客数・推計）",
    "客単価（税抜）","客単価（税込）",
    "店舗名（予約）","商品カテゴリ（予約）"
]
STATE_HEADER = ["店舗名", "最終取得日", "最終更新日時"]  # 最終取得日=YYYY/MM/DD

def get_sheets_service():
    creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    return build("sheets", "v4", credentials=creds)

def get_sheet_titles_and_ids():
    service = get_sheets_service()
    meta = service.spreadsheets().get(spreadsheetId=SPREADSHEET_ID).execute()
    out = {}
    for s in meta.get("sheets", []):
        prop = s.get("properties", {})
        out[prop.get("title")] = prop.get("sheetId")
    return out

def ensure_sheet_exists(title):
    titles = get_sheet_titles_and_ids()
    if title in titles:
        return
    service = get_sheets_service()
    service.spreadsheets().batchUpdate(
        spreadsheetId=SPREADSHEET_ID,
        body={"requests": [{"addSheet": {"properties": {"title": title}}}]}
    ).execute()
    print(f"✅ シート作成: {title}")

def ensure_header(sheet_name, header_row):
    service = get_sheets_service()
    res = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=f"{sheet_name}!A1:Z1"
    ).execute()
    values = res.get("values", [])
    if (not values) or (not values[0]) or (str(values[0][0]).strip() == ""):
        service.spreadsheets().values().update(
            spreadsheetId=SPREADSHEET_ID,
            range=f"{sheet_name}!A1",
            valueInputOption="RAW",
            body={"values": [header_row]}
        ).execute()
        print(f"✅ ヘッダー投入: {sheet_name}")

def set_fact_date_formats():
    """
    Looker Studio が日付認定しなくなる対策：
    Fact_時間別 の A列(日付) と B列(月) を「日付フォーマット」に固定。
    B列は「yyyy-MM」表示にする（実体は日付：月初日）。
    """
    ids = get_sheet_titles_and_ids()
    sid = ids.get(SHEET_FACT)
    if sid is None:
        return

    service = get_sheets_service()

    reqs = [
        {
            "repeatCell": {
                "range": {"sheetId": sid, "startRowIndex": 1, "startColumnIndex": 0, "endColumnIndex": 1},
                "cell": {"userEnteredFormat": {"numberFormat": {"type": "DATE", "pattern": "yyyy/MM/dd"}}},
                "fields": "userEnteredFormat.numberFormat"
            }
        },
        {
            "repeatCell": {
                "range": {"sheetId": sid, "startRowIndex": 1, "startColumnIndex": 1, "endColumnIndex": 2},
                "cell": {"userEnteredFormat": {"numberFormat": {"type": "DATE", "pattern": "yyyy-MM"}}},
                "fields": "userEnteredFormat.numberFormat"
            }
        }
    ]
    service.spreadsheets().batchUpdate(
        spreadsheetId=SPREADSHEET_ID,
        body={"requests": reqs}
    ).execute()
    print("✅ Fact_時間別: A列/B列 の日付表示形式を固定しました")

def chunked(rows, size):
    for i in range(0, len(rows), size):
        yield rows[i:i+size]

def append_rows(sheet_name, rows, chunk_size=3000):
    if not rows:
        return 0
    service = get_sheets_service()
    total = 0
    for part in chunked(rows, chunk_size):
        service.spreadsheets().values().append(
            spreadsheetId=SPREADSHEET_ID,
            range=f"{sheet_name}!A1",
            valueInputOption="USER_ENTERED",
            insertDataOption="INSERT_ROWS",
            body={"values": part}
        ).execute()
        total += len(part)
    return total

def read_state_latest_dates():
    service = get_sheets_service()
    res = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=f"{SHEET_STATE}!A2:C"
    ).execute()
    rows = res.get("values", [])
    latest = {}
    for r in rows:
        if len(r) < 2:
            continue
        store = str(r[0]).strip()
        dstr  = str(r[1]).strip()
        if not store or not dstr:
            continue
        try:
            latest[store] = datetime.strptime(dstr, "%Y/%m/%d")
        except:
            pass
    return latest

def write_state_bulk(latest_by_store):
    service = get_sheets_service()
    now_str = datetime.now().strftime("%Y/%m/%d %H:%M:%S")
    values = [STATE_HEADER]
    for store, d in sorted(latest_by_store.items(), key=lambda x: x[0]):
        values.append([store, d.strftime("%Y/%m/%d"), now_str])

    service.spreadsheets().values().clear(
        spreadsheetId=SPREADSHEET_ID,
        range=f"{SHEET_STATE}!A:C"
    ).execute()
    service.spreadsheets().values().update(
        spreadsheetId=SPREADSHEET_ID,
        range=f"{SHEET_STATE}!A1",
        valueInputOption="RAW",
        body={"values": values}
    ).execute()
    print("✅ 取得管理 更新完了")

def init_state_from_fact_if_needed(existing_state):
    if existing_state:
        return existing_state

    print("ℹ️ 取得管理が空のため、初回だけ Fact_時間別 を走査して初期化します（ここだけ重い）")
    service = get_sheets_service()
    res = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=f"{SHEET_FACT}!A:K"
    ).execute()
    rows = res.get("values", [])
    if len(rows) <= 1:
        return {}

    latest = {}
    for r in rows[1:]:
        if len(r) < 11:
            continue
        date_str = str(r[0]).strip()
        store    = str(r[10]).strip()
        if not date_str or not store:
            continue
        try:
            d = datetime.strptime(date_str, "%Y/%m/%d")
        except:
            continue
        if store not in latest or d > latest[store]:
            latest[store] = d

    if latest:
        write_state_bulk(latest)
    return latest

def load_existing_keys_for_range(store_name, from_date, to_date):
    """
    既存重複排除：店舗×日付×時間（A, D, K）
    ※全件読みが重いなら後で最適化する（まず動かす優先）
    """
    service = get_sheets_service()
    res = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=f"{SHEET_FACT}!A:K"
    ).execute()
    rows = res.get("values", [])
    if len(rows) <= 1:
        return set()

    s_from = from_date.strftime("%Y/%m/%d")
    s_to   = to_date.strftime("%Y/%m/%d")

    keys = set()
    for r in rows[1:]:
        if len(r) < 11:
            continue
        d  = str(r[0]).strip()
        h  = str(r[3]).strip()
        st = str(r[10]).strip()
        if st != store_name:
            continue
        if d < s_from or d > s_to:
            continue
        keys.add((st, d, h))
    return keys


# ============================================================
# 2) 共通ユーティリティ（曜日/時間帯/数値）
# ============================================================

HOURS = list(range(24))
TAX_RATE = 0.10

def get_weekday_label(dt):
    wd = ["月","火","水","木","金","土","日"][dt.weekday()]
    if jpholiday.is_holiday(dt):
        return "祝"
    return wd

def get_part_of_day(hour):
    return "ランチ" if hour <= 15 else "ディナー"

def month_first_str(dt: datetime) -> str:
    # B列は「月初日」を入れて、表示形式で yyyy-MM にする
    return datetime(dt.year, dt.month, 1).strftime("%Y/%m/%d")

def parse_int(text):
    t = str(text or "").replace(",", "").strip()
    return int(t) if t else 0

def to_number_safe(s) -> float:
    if s is None:
        return 0.0
    t = str(s).strip()
    if not t:
        return 0.0
    for ch in [",", "円", "￥", "\\", " "]:
        t = t.replace(ch, "")
    t = t.replace("－", "-").replace("―", "-")
    if t in ["-", ""]:
        return 0.0
    if not re.fullmatch(r"-?\d+(\.\d+)?", t):
        m = re.findall(r"\d+", t)
        if not m:
            return 0.0
        t = "".join(m)
    try:
        return float(t)
    except:
        return 0.0

def round0(x: float) -> float:
    return float(int(round(float(x), 0)))

def to_tax_excluded_round0(tax_in: float) -> float:
    return round0(float(tax_in) / (1.0 + TAX_RATE))


# ============================================================
# 3) Blayn（requests）設定 & 実装  ※ID/PWはENVから
# ============================================================

BLAYN_LOGIN_URL = "https://secure.blayn.com/mng/account/login"
BLAYN_TIME_URL  = "https://secure.blayn.com/mng/sales/time?t=d&date={date}"
BLAYN_ITEM_URL  = "https://secure.blayn.com/mng/summary/item?t=d&date={date}"

BLAYN_STORES = [
    {
        "name": "鶏のとりこ",
        "username": env_required("BLAYN_TORIKO_USER"),
        "password": env_required("BLAYN_TORIKO_PASS"),
        "first_from": datetime(2025, 12, 9),
    },
    {
        "name": "かわさ鬼",
        "username": env_required("BLAYN_KAWASAKI_USER"),
        "password": env_required("BLAYN_KAWASAKI_PASS"),
        "first_from": datetime(2025, 7, 22),
    },
]

def make_session():
    s = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

def blayn_login(session: requests.Session, username: str, password: str) -> bool:
    payload = {
        "login_nm": username,
        "login_pw": password,
        "save": "true",
        "submit[login]": "ログイン"
    }
    res = session.post(BLAYN_LOGIN_URL, data=payload, timeout=30)
    return ("ログアウト" in res.text)

def run_blayn(state_latest: dict, to_date: datetime):
    print("\n====================")
    print("🚀 RUN: Blayn")
    print("====================")

    fact_rows = []
    new_latest = dict(state_latest)

    for store in BLAYN_STORES:
        store_name = store["name"]
        username   = store["username"]
        password   = store["password"]
        first_from = store["first_from"]

        if store_name in state_latest:
            from_date = state_latest[store_name] + timedelta(days=1)
        else:
            from_date = first_from

        if from_date > to_date:
            print(f"🟦 {store_name}: 差分なし（取得スキップ）")
            continue

        print(f"🟩 {store_name}: 取得範囲 {from_date:%Y/%m/%d} → {to_date:%Y/%m/%d}")

        existing_keys = load_existing_keys_for_range(store_name, from_date, to_date)

        session = make_session()
        if not blayn_login(session, username, password):
            print(f"❌ {store_name}: ログイン失敗（ID/PW確認）")
            continue

        cur = from_date
        max_success = None

        while cur <= to_date:
            date_param = cur.strftime("%Y%m%d")
            date_str   = cur.strftime("%Y/%m/%d")
            month_str  = month_first_str(cur)
            wd_lbl     = get_weekday_label(cur)

            hourly_sales_incl = [0] * 24
            hourly_sales_excl = [0] * 24
            day_total_sales_excl = 0
            ramen_qty_today = 0
            time_ok = False

            # ① 時間帯別売上
            try:
                res_time = session.get(BLAYN_TIME_URL.format(date=date_param), timeout=30)
                soup_time = BeautifulSoup(res_time.text, "lxml")

                canvas = soup_time.find("canvas", {"id": "chartGraph"})
                if canvas is None or not canvas.has_attr("data-sales"):
                    raise ValueError("data-sales が見つかりません")

                sales_data = list(map(int, canvas["data-sales"].split(",")))
                if len(sales_data) != 24:
                    raise ValueError("24時間分のデータがありません")

                for h in HOURS:
                    s_incl = sales_data[h]
                    s_excl = int(round(s_incl / 1.1))
                    hourly_sales_incl[h] = s_incl
                    hourly_sales_excl[h] = s_excl

                day_total_sales_excl = sum(hourly_sales_excl)
                time_ok = True
            except Exception as e:
                print(f"  ⚠️ {store_name} {date_str}: 時間帯別売上 失敗 -> {e}")

            # ② ABC（ラーメン杯数のみ）
            try:
                res_item = session.get(BLAYN_ITEM_URL.format(date=date_param), timeout=30)
                soup_item = BeautifulSoup(res_item.text, "lxml")

                for tr in soup_item.select("tbody tr"):
                    tds = tr.find_all("td")
                    if len(tds) < 2:
                        continue

                    first_td = tds[0]
                    span = first_td.find("span")
                    if span and span.has_attr("data-full"):
                        item_name = span["data-full"].strip()
                    else:
                        item_name = first_td.get_text(strip=True)

                    if not item_name:
                        continue

                    qty = parse_int(tds[1].get_text(strip=True))
                    if "ラーメン" in item_name:
                        ramen_qty_today += qty
            except Exception as e:
                print(f"  ⚠️ {store_name} {date_str}: ABC 失敗 -> {e}")

            # ③ Fact行作成（重複キーは弾く）
            if time_ok:
                ramen_by_hour = [0.0] * 24
                if ramen_qty_today > 0 and day_total_sales_excl > 0:
                    for h in HOURS:
                        share_h = hourly_sales_excl[h] / day_total_sales_excl if day_total_sales_excl > 0 else 0
                        ramen_by_hour[h] = ramen_qty_today * share_h

                added_any = False

                for h in HOURS:
                    h_str = f"{h:02d}:00"
                    key = (store_name, date_str, h_str)
                    if key in existing_keys:
                        continue

                    s_incl = float(hourly_sales_incl[h])
                    s_excl = float(hourly_sales_excl[h])
                    ramen_h = float(ramen_by_hour[h])

                    if ramen_h > 0:
                        unit_ex = round0(s_excl / ramen_h)
                        unit_in = round0(s_incl / ramen_h)
                    else:
                        unit_ex = 0.0
                        unit_in = 0.0

                    part = get_part_of_day(h)

                    fact_rows.append([
                        date_str, month_str, wd_lbl, h_str, part,
                        s_incl, s_excl,
                        ramen_h,
                        unit_ex, unit_in,
                        store_name, ""
                    ])

                    existing_keys.add(key)
                    added_any = True

                if added_any:
                    if (max_success is None) or (cur > max_success):
                        max_success = cur

            cur += timedelta(days=1)

        if max_success is not None:
            prev = new_latest.get(store_name)
            if (prev is None) or (max_success > prev):
                new_latest[store_name] = max_success

    added = append_rows(SHEET_FACT, fact_rows, chunk_size=3000)
    print(f"📊 Blayn: Fact_時間別 追記 {added}行")

    return new_latest


# ============================================================
# 4) ticket.shiagel.jp（Playwright）【Colabで動いたロジックを移植】
# ============================================================

TICKET_LOGIN_EMAIL    = env_required("TICKET_LOGIN_EMAIL")
TICKET_LOGIN_PASSWORD = env_required("TICKET_LOGIN_PASSWORD")

TICKET_URL_LOGIN       = env_required("TICKET_URL_LOGIN")        # 例: https://ticket.shiagel.jp/GID001M101
TICKET_URL_SALES_DAILY = env_required("TICKET_URL_SALES_DAILY")  # 例: https://ticket.shiagel.jp/GID008M301

TICKET_COMPANY_NAME = os.environ.get("TICKET_COMPANY_NAME", "")  # 任意（無くても動くようにする）

TICKET_FIRST_FROM_FIXED = datetime(2025, 12, 23)

TICKET_STORES = [
    {"store_key": "麺屋周郷 別邸 雅", "store_contains": "麺屋周郷 別邸 雅", "first_from": TICKET_FIRST_FROM_FIXED},
]

STEP_TIMEOUT_MS = 45000
DATE_TIMEOUT_S  = 90

def normalize_ymd(s: str) -> str:
    s = str(s or "")
    m = re.search(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})", s)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04d}/{mo:02d}/{d:02d}"
    m = re.search(r"(\d{4})\.(\d{1,2})\.(\d{1,2})", s)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04d}/{mo:02d}/{d:02d}"
    m = re.search(r"(\d{4})\s*年\s*(\d{1,2})\s*月\s*(\d{1,2})\s*日", s)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04d}/{mo:02d}/{d:02d}"
    raise ValueError(f"日付形式が不正: {s}")

def extract_ymd_from_text_any(s: str):
    s = str(s or "")
    m = re.search(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})", s)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04d}/{mo:02d}/{d:02d}"
    m = re.search(r"(\d{4})\.(\d{1,2})\.(\d{1,2})", s)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04d}/{mo:02d}/{d:02d}"
    m = re.search(r"(\d{4})\s*年\s*(\d{1,2})\s*月\s*(\d{1,2})\s*日", s)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04d}/{mo:02d}/{d:02d}"
    return None

def make_zero_24h():
    return [{"hour": hh, "sales_in": 0.0, "sales_ex": 0.0, "guests": 0.0} for hh in range(24)]

async def safe_wait(awaitable, label):
    try:
        return await asyncio.wait_for(awaitable, timeout=STEP_TIMEOUT_MS/1000)
    except asyncio.TimeoutError:
        raise TimeoutError(f"TIMEOUT at {label}")

async def _first_locator(page, selectors):
    for sel in selectors:
        loc = page.locator(sel)
        try:
            if await loc.count() > 0:
                return loc
        except:
            pass
    return None

async def ticket_login_if_needed(page):
    await page.goto(TICKET_URL_LOGIN, wait_until="domcontentloaded", timeout=STEP_TIMEOUT_MS)

    email_loc = await _first_locator(page, [
        "input[placeholder='メールアドレス']",
        "input[type='email']",
        "input[name='email']",
        "input[autocomplete='username']",
        "input[placeholder*='メール']",
    ])
    pass_loc = await _first_locator(page, [
        "input[placeholder='パスワード']",
        "input[type='password']",
        "input[name='password']",
        "input[autocomplete='current-password']",
        "input[placeholder*='パスワード']",
    ])

    # 既にログイン済っぽいならスキップ
    if (email_loc is None) or (pass_loc is None):
        return

    await email_loc.first.fill(TICKET_LOGIN_EMAIL)
    await pass_loc.first.fill(TICKET_LOGIN_PASSWORD)

    btn = await _first_locator(page, [
        "button:has-text('ログイン')",
        "button[type='submit']",
        "input[type='submit']",
        "text=ログイン",
    ])
    if btn is None:
        raise RuntimeError("ログインボタンが見つかりません（セレクタ要調整）")

    await btn.first.click()
    try:
        await page.wait_for_load_state("networkidle", timeout=STEP_TIMEOUT_MS)
    except:
        await page.wait_for_timeout(1200)

async def reach_sales_daily(page):
    # ログイン保証 → 売上（日次）へ
    await ticket_login_if_needed(page)
    await page.goto(TICKET_URL_SALES_DAILY, wait_until="domcontentloaded", timeout=STEP_TIMEOUT_MS)
    try:
        await page.wait_for_load_state("networkidle", timeout=STEP_TIMEOUT_MS)
    except:
        await page.wait_for_timeout(1200)

# ---------- 店舗選択（Colab版と同じ：select[0]のoptionsを走査） ----------
async def ensure_store_selected(page, store_contains: str):
    await safe_wait(page.wait_for_selector("select"), "store select exists")
    await safe_wait(page.wait_for_function("""() => {
      const s0 = document.querySelectorAll('select')[0];
      return s0 && s0.options && s0.options.length > 0;
    }"""), "store options injected")

    opts = await page.evaluate("""() => {
      const s0 = document.querySelectorAll('select')[0];
      return Array.from(s0.options).map(o => ({value:o.value, text:(o.textContent||'').trim()}));
    }""")
    if not opts:
        raise RuntimeError("店舗select optionsが空です")

    target = None
    for o in opts:
        if store_contains and store_contains in o["text"]:
            target = o["value"]
            break
    if target is None:
        target = opts[0]["value"]

    await page.locator("select").nth(0).select_option(value=target)
    await page.wait_for_timeout(200)

# ---------- 日次ラジオ（filter_radio3） ----------
async def select_daily_radio(page):
    await safe_wait(page.wait_for_selector("xpath=//input[@id='filter_radio3']", state="attached"), "radio attach")

    label = page.locator("xpath=//label[@for='filter_radio3']")
    if await label.count() > 0:
        try:
            await label.first.click(timeout=15000)
            await page.wait_for_timeout(150)
            return
        except:
            pass

    radio = page.locator("xpath=//input[@id='filter_radio3']")
    await radio.check(force=True, timeout=15000)
    await page.wait_for_timeout(150)

# ---------- v3dp（日付クリック） ----------
XP_V3DP_POPOUT_VISIBLE = "//div[contains(@class,'v3dp__popout') and not(contains(@style,'display: none'))]"
XP_V3DP_HEADING_CENTER_VISIBLE = XP_V3DP_POPOUT_VISIBLE + "//div[contains(@class,'v3dp__heading')]//*[self::button or self::span][contains(@class,'v3dp__heading__center')]"
XP_V3DP_PREV_VISIBLE = XP_V3DP_POPOUT_VISIBLE + "//div[contains(@class,'v3dp__heading')]//button[contains(@class,'v3dp__heading__button')][1]"
XP_V3DP_NEXT_VISIBLE = XP_V3DP_POPOUT_VISIBLE + "//div[contains(@class,'v3dp__heading')]//button[contains(@class,'v3dp__heading__button')][last()]"

XP_DAILY_FROM_INPUT = "//label[@for='filter_radio3']//span[contains(@class,'input_date')][1]//input[contains(@class,'w160')]"
XP_DAILY_TO_INPUT   = "//label[@for='filter_radio3']//span[contains(@class,'input_date')][2]//input[contains(@class,'w160')]"

def _parse_heading_jp_month_year(s: str):
    s = (s or "").replace("　", " ").strip()
    m = re.search(r"(\d{1,2})\s*月\s*(\d{4})", s)
    if m:
        mo = int(m.group(1))
        y  = int(m.group(2))
        return y, mo
    parts = s.split()
    if len(parts) >= 2 and "月" in parts[0]:
        mo = int(parts[0].replace("月", ""))
        y  = int(parts[1].replace("年", ""))
        return y, mo
    raise ValueError(f"カレンダーヘッダー形式が想定外: {s}")

async def _get_calendar_ym(page):
    await page.wait_for_selector(f"xpath={XP_V3DP_HEADING_CENTER_VISIBLE}")
    s = (await page.locator(f"xpath={XP_V3DP_HEADING_CENTER_VISIBLE}").inner_text()).strip()
    y, mo = _parse_heading_jp_month_year(s)
    return y * 12 + mo

async def _ensure_calendar_month(page, target_dt: datetime):
    targetYM = target_dt.year * 12 + target_dt.month
    for _ in range(48):
        curYM = await _get_calendar_ym(page)
        if curYM == targetYM:
            return
        if curYM > targetYM:
            await page.locator(f"xpath={XP_V3DP_PREV_VISIBLE}").click()
        else:
            await page.locator(f"xpath={XP_V3DP_NEXT_VISIBLE}").click()
        await page.wait_for_timeout(120)
    raise RuntimeError(f"カレンダー月移動が上限: target={target_dt:%Y/%m}")

async def _pick_one_date_by_click(page, xp_input: str, target_dt: datetime):
    await page.wait_for_selector(f"xpath={xp_input}")
    await page.locator(f"xpath={xp_input}").click()
    await page.wait_for_timeout(120)

    if await page.locator(f"xpath={XP_V3DP_POPOUT_VISIBLE}").count() == 0:
        await page.locator(f"xpath={xp_input}").click()
        await page.wait_for_timeout(150)

    await page.wait_for_selector(f"xpath={XP_V3DP_POPOUT_VISIBLE}")
    await _ensure_calendar_month(page, target_dt)

    dd0 = f"{target_dt.day:02d}"
    dd1 = str(target_dt.day)
    xp_day = (
        XP_V3DP_POPOUT_VISIBLE
        + "//div[contains(@class,'v3dp__elements')]//button[not(@disabled)]"
        + f"[.//span[normalize-space(.)='{dd0}' or normalize-space(.)='{dd1}']]"
    )
    await page.wait_for_selector(f"xpath={xp_day}")
    await page.locator(f"xpath={xp_day}").click()
    await page.wait_for_timeout(120)

async def set_daily_date_range_click(page, ymd: str):
    ymd = normalize_ymd(ymd)
    dt = datetime.strptime(ymd, "%Y/%m/%d")
    await _pick_one_date_by_click(page, XP_DAILY_FROM_INPUT, dt)
    await _pick_one_date_by_click(page, XP_DAILY_TO_INPUT, dt)
    await page.wait_for_timeout(150)

# ---------- 検索 & 更新待ち ----------
async def click_search_and_wait(page, target_ymd: str):
    target_ymd = normalize_ymd(target_ymd)

    await safe_wait(page.wait_for_selector(
        "xpath=//p[contains(@class,'btn_search')]//button[normalize-space()='検索']"
    ), "search button")

    before = await page.evaluate("""() => {
      const tr = document.querySelector("table tbody tr");
      return tr ? (tr.textContent||"").trim() : "";
    }""")

    await page.click("xpath=//p[contains(@class,'btn_search')]//button[normalize-space()='検索']")

    await safe_wait(page.wait_for_function("""
    ({target, before}) => {
      const msg = document.querySelector("ul.error_message li")?.textContent || "";
      if (msg.includes("データがありません")) return true;

      const trs = Array.from(document.querySelectorAll("table tbody tr"));
      if (!trs.length) return false;

      const first = (trs[0].textContent||"").trim();
      if (before && first && first !== before) return true;

      const any = trs.some(tr => (tr.textContent||"").includes(target));
      return any;
    }
    """, arg={"target": target_ymd, "before": before}), "search result updated")

    await page.wait_for_timeout(200)

async def is_no_data(page) -> bool:
    return await page.locator(
        "xpath=//ul[contains(@class,'error_message')]/li[contains(.,'データがありません')]"
    ).count() > 0

# ---------- 日次表ダンプ ----------
async def get_daily_rows_dump(page):
    return await page.evaluate("""() => {
      const trs = Array.from(document.querySelectorAll("table tbody tr"));
      return trs.map((tr, idx) => {
        const tds = Array.from(tr.querySelectorAll("td")).map(td => (td.innerText||td.textContent||"").trim());
        const links = Array.from(tr.querySelectorAll("a")).map(a => ({
          text: (a.innerText||a.textContent||"").trim(),
          href: a.getAttribute("href") || ""
        }));
        return {
          idx,
          textContent: (tr.innerText||tr.textContent||"").trim(),
          tds,
          links
        };
      });
    }""")

def extract_ymd_from_row_dump(row):
    candidates = []
    if row.get("textContent"):
        candidates.append(row["textContent"])
    for td in row.get("tds") or []:
        if td:
            candidates.append(td)
    for c in candidates:
        ymd = extract_ymd_from_text_any(c)
        if ymd:
            return ymd
    return None

# ---------- 時間帯別へ（href優先） ----------
async def open_hourly_from_daily_dump(page, target_ymd: str, dump_rows):
    target_norm = normalize_ymd(target_ymd)

    hit = None
    for r in dump_rows:
        ymd = extract_ymd_from_row_dump(r)
        if ymd == target_norm:
            hit = r
            break

    if hit is None and len(dump_rows) == 1:
        hit = dump_rows[0]

    if hit is None:
        return False

    row_locator = page.locator("xpath=//table//tbody//tr").nth(hit["idx"])

    link = row_locator.locator("xpath=.//a[contains(@href,'GID008M307')]")
    if await link.count() == 0:
        link = row_locator.locator("xpath=.//a[contains(normalize-space(.),'時間帯')]")

    if await link.count() == 0:
        return False

    await link.first.click()
    await safe_wait(page.wait_for_url("**/GID008M307**"), "hourly page")
    return True

# ---------- 時間帯別 ready ----------
async def wait_hourly_page_ready(page, timeout_sec=25):
    await page.wait_for_timeout(200)
    tb = page.locator("xpath=//tbody[@aria-live='polite' and .//tr[@role='row']]")
    await tb.wait_for(state="attached", timeout=timeout_sec*1000)

    first_cell = page.locator("xpath=//tbody[@aria-live='polite']//tr[@role='row'][1]/td[1]")
    await first_cell.wait_for(state="attached", timeout=timeout_sec*1000)

    await page.wait_for_function(
        """() => {
          const el = document.querySelector("tbody[aria-live='polite'] tr[role='row'] td");
          if(!el) return false;
          const t = (el.textContent || "").trim();
          return t.includes(":00") || t.includes("時間外");
        }""",
        timeout=timeout_sec*1000
    )

# ---------- 時間帯別テーブル parse（Colab版そのまま） ----------
async def parse_hourly_table(page):
    data = make_zero_24h()

    tr_locator = page.locator("xpath=//tbody[@aria-live='polite']//tr[@role='row']")
    n = await tr_locator.count()
    if n == 0:
        tr_locator = page.locator("xpath=//table//tbody//tr")
        n = await tr_locator.count()

    for i in range(n):
        tr = tr_locator.nth(i)
        tds = tr.locator("td")
        td_count = await tds.count()
        if td_count < 3:
            continue

        slot_text = (await tds.nth(0).inner_text()).strip()
        if (not slot_text) or ("時間外" in slot_text):
            continue

        m = re.search(r"(\d{1,2})\s*:", slot_text)
        if not m:
            continue
        hh = int(m.group(1))
        if hh < 0 or hh > 23:
            continue

        sales_in = 0.0
        guests   = 0.0

        if td_count >= 11:
            sales_in = to_number_safe(await tds.nth(2).inner_text())  # 3列目
            guests   = to_number_safe(await tds.nth(9).inner_text())  # 10列目
        else:
            sales_in = to_number_safe(await tds.nth(2).inner_text())
            guests   = 0.0

        data[hh]["sales_in"] = float(sales_in)
        data[hh]["sales_ex"] = float(to_tax_excluded_round0(sales_in))
        data[hh]["guests"]   = float(guests)

    return data

# ---------- 日次へ戻る ----------
async def back_to_daily(page):
    try:
        back = page.locator("xpath=//p[contains(@class,'link_back')]//a[contains(.,'全体売上') or contains(.,'戻る')]")
        if await back.count() > 0:
            await back.first.click()
            await safe_wait(page.wait_for_url("**/GID008M301**"), "back daily")
            await page.wait_for_timeout(200)
            return
    except:
        pass

    await page.goto(TICKET_URL_SALES_DAILY, wait_until="domcontentloaded")
    await page.wait_for_timeout(200)

def _is_all_zero(hourly):
    return all((float(x.get("sales_in",0))==0 and float(x.get("sales_ex",0))==0 and float(x.get("guests",0))==0) for x in hourly)

# ---------- 1日分（VBA導線） ----------
async def fetch_one_day_hourly_ticket(page, store_contains: str, target_ymd: str, debug=False):
    target_ymd = normalize_ymd(target_ymd)

    async def _inner():
        await ensure_store_selected(page, store_contains)
        await select_daily_radio(page)
        await set_daily_date_range_click(page, target_ymd)
        await click_search_and_wait(page, target_ymd)

        if await is_no_data(page):
            if debug:
                print(f"  🟨 {target_ymd}: データがありません（no_data）")
            return make_zero_24h()

        dump = await get_daily_rows_dump(page)
        ok = await open_hourly_from_daily_dump(page, target_ymd, dump)
        if not ok:
            if debug:
                shown = []
                for r in dump[:30]:
                    y = extract_ymd_from_row_dump(r)
                    if y:
                        shown.append(y)
                print(f"  🟥 {target_ymd}: 時間帯別へ遷移できず（候補: {shown[:15]}）")
            return make_zero_24h()

        await wait_hourly_page_ready(page, timeout_sec=25)
        hourly = await parse_hourly_table(page)
        await back_to_daily(page)

        # デバッグ：全部0ならスクショ/HTMLを保存（Actionsのartifactに入れる）
        if debug and _is_all_zero(hourly):
            os.makedirs("out", exist_ok=True)
            safe_date = target_ymd.replace("/", "")
            png = f"out/ticket_{store_contains}_{safe_date}.png"
            html = f"out/ticket_{store_contains}_{safe_date}.html"
            await page.screenshot(path=png, full_page=True)
            with open(html, "w", encoding="utf-8") as f:
                f.write(await page.content())
            print(f"🧾 DEBUG SAVED: {png}, {html}")

        return hourly

    try:
        return await asyncio.wait_for(_inner(), timeout=DATE_TIMEOUT_S)
    except asyncio.TimeoutError:
        print(f"  ⚠️ {target_ymd}: 1日処理タイムアウト → 日次へ戻して継続")
        try:
            await page.goto(TICKET_URL_SALES_DAILY, wait_until="domcontentloaded")
            await page.wait_for_timeout(400)
        except:
            pass
        return make_zero_24h()

async def run_ticket(state_latest: dict, to_date_global: datetime, flush_every_days: int = 3):
    print("\n====================")
    print("🚀 RUN: ticket.shiagel.jp")
    print("====================")

    new_latest = dict(state_latest)

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        page = await browser.new_page()

        try:
            await reach_sales_daily(page)

            for store in TICKET_STORES:
                store_key      = store["store_key"]
                store_contains = store["store_contains"]
                first_from     = store["first_from"]

                if store_key in state_latest:
                    from_date = state_latest[store_key] + timedelta(days=1)
                else:
                    from_date = first_from

                to_date = to_date_global

                if from_date > to_date:
                    print(f"🟦 {store_key}: 差分なし（取得スキップ）")
                    continue

                print(f"🟩 {store_key}: 取得範囲 {from_date:%Y/%m/%d} → {to_date:%Y/%m/%d}")

                existing_keys = load_existing_keys_for_range(store_key, from_date, to_date)

                cur = from_date
                buffer_rows = []
                days_since_flush = 0

                while cur <= to_date:
                    date_str  = cur.strftime("%Y/%m/%d")
                    month_str = month_first_str(cur)
                    wd_lbl    = get_weekday_label(cur)

                    hourly = await fetch_one_day_hourly_ticket(page, store_contains, date_str, debug=True)

                    added_any = False
                    for hh in range(24):
                        h_str = f"{hh:02d}:00"
                        key = (store_key, date_str, h_str)
                        if key in existing_keys:
                            continue

                        part = get_part_of_day(hh)
                        sales_in = float(hourly[hh]["sales_in"])
                        sales_ex = float(hourly[hh]["sales_ex"])
                        guests   = float(hourly[hh]["guests"])

                        if guests > 0:
                            unit_in = round0(sales_in / guests)
                            unit_ex = round0(sales_ex / guests)
                        else:
                            unit_in = 0.0
                            unit_ex = 0.0

                        ramen_cnt = guests  # 仕様：客数=杯数扱い

                        buffer_rows.append([
                            date_str, month_str, wd_lbl, h_str, part,
                            sales_in, sales_ex,
                            ramen_cnt,
                            unit_ex, unit_in,
                            store_key, ""
                        ])

                        existing_keys.add(key)
                        added_any = True

                    if added_any:
                        new_latest[store_key] = cur
                    else:
                        print(f"  🟦 {store_key} {date_str}: 追記なし（既存キーでスキップ）")

                    days_since_flush += 1
                    if (days_since_flush >= flush_every_days) and buffer_rows:
                        added = append_rows(SHEET_FACT, buffer_rows, chunk_size=2000)
                        print(f"  ✅ {store_key}: flush追記 {added}行（{days_since_flush}日分）")
                        buffer_rows = []
                        days_since_flush = 0
                        write_state_bulk(new_latest)

                    cur += timedelta(days=1)

                if buffer_rows:
                    added = append_rows(SHEET_FACT, buffer_rows, chunk_size=2000)
                    print(f"  ✅ {store_key}: 最終flush追記 {added}行")
                    write_state_bulk(new_latest)

        finally:
            await browser.close()

    return new_latest

# ============================================================
# 5) メイン実行（Blayn → ticket → state更新）
# ============================================================

async def main():
    ensure_sheet_exists(SHEET_STATE)
    ensure_sheet_exists(SHEET_FACT)
    ensure_header(SHEET_STATE, STATE_HEADER)
    ensure_header(SHEET_FACT, FACT_HEADER)

    # 日付認定崩れ対策（おすすめ）
    set_fact_date_formats()

    state_latest = read_state_latest_dates()
    state_latest = init_state_from_fact_if_needed(state_latest)

    today = datetime.today()
    today0 = datetime(today.year, today.month, today.day)
    yesterday0 = today0 - timedelta(days=1)

    print("\n====================")
    print("🧭 Global Range")
    print("====================")
    print("today0     :", today0.strftime("%Y/%m/%d %H:%M:%S"))
    print("yesterday0 :", yesterday0.strftime("%Y/%m/%d %H:%M:%S"))

    state_after_blayn = run_blayn(state_latest, to_date=yesterday0)

    # ★Notebook(Papermill)はこれが正解：await で呼ぶ
    state_after_ticket = await run_ticket(
        state_after_blayn, to_date_global=yesterday0, flush_every_days=3
    )

    write_state_bulk(state_after_ticket)
    print("\n✅ 全部完了（Blayn + ticket 統合 / Actions対応 ）")

# ★Notebookのセル末尾はこれ（asyncio.run は絶対に使わない）
await main()
